# Interactive Visual Analytics with Folium

Launch-site circles and markers, red/green outcome clusters, mouse coordinates and proximity lines support geographic inspection. Proximity calculations use the nearest mapped coastline, rail and public-road line segments from OpenStreetMap. They are straight-line distances, not travel distances.

Prepared with AI assistance. Data and outputs are reproducible.

In [1]:
"""Reproducible IBM Applied Data Science Capstone analysis.
Run: python analysis.py. Historical course snapshots are included in data/.
"""
from pathlib import Path
import json, sqlite3, io, contextlib, base64
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import folium
from folium.plugins import MarkerCluster, MousePosition
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, balanced_accuracy_score
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
DATA=ROOT/'data'; RESULTS=ROOT/'results'; RESULTS.mkdir(exist_ok=True)
plt.rcParams.update({'font.size':12,'axes.spines.top':False,'axes.spines.right':False})


def map_analysis():
    df=pd.read_csv(DATA/'spacex_launch_geo.csv'); m=folium.Map(location=[30,-90],zoom_start=4,tiles='OpenStreetMap')
    sites=df.groupby('Launch Site')[['Lat','Long']].first()
    for site,r in sites.iterrows():
        folium.Circle([r.Lat,r.Long],radius=1000,color='#087f8c',fill=True,popup=site).add_to(m)
        folium.Marker([r.Lat,r.Long],tooltip=site,popup=site).add_to(m)
    cluster=MarkerCluster(name='Launch outcomes').add_to(m)
    for _,r in df.iterrows():
        folium.Marker([r.Lat,r.Long],popup=f"Flight {r['Flight Number']} | {r['Date']} | class={r['class']}",icon=folium.Icon(color='green' if r['class']==1 else 'red')).add_to(cluster)
    MousePosition(position='topright',separator=' | ',prefix='Latitude / longitude').add_to(m)
    folium.LayerControl().add_to(m); m.save(ROOT/'launch_sites_map.html')
    # Nearest points on mapped OSM line segments, using a local planar projection.
    site=sites.loc['CCAFS SLC-40']; import math, xml.etree.ElementTree as ET
    tree=ET.parse(DATA/'osm_proximity.xml').getroot()
    nodes={n.attrib['id']:(float(n.attrib['lat']),float(n.attrib['lon'])) for n in tree.findall('node')}
    nearest={}; feature_meta={}; lat0=float(site.Lat); lon0=float(site.Long); cosine=math.cos(math.radians(lat0))
    for way in tree.findall('way'):
        tags={t.attrib['k']:t.attrib['v'] for t in way.findall('tag')}
        label='Coastline' if tags.get('natural')=='coastline' else ('Railway' if tags.get('railway')=='rail' else ('Road' if tags.get('highway') in ['primary','secondary','tertiary','unclassified','residential'] else None))
        if not label:continue
        pts=[nodes[n.attrib['ref']] for n in way.findall('nd') if n.attrib['ref'] in nodes]
        for (la,lo),(lb,lob) in zip(pts,pts[1:]):
            ax=(lo-lon0)*cosine;ay=la-lat0;bx=(lob-lon0)*cosine;by=lb-lat0
            dx=bx-ax;dy=by-ay; den=dx*dx+dy*dy
            t=max(0,min(1,-(ax*dx+ay*dy)/den)) if den else 0
            x=ax+t*dx;y=ay+t*dy;dist=x*x+y*y
            if label not in nearest or dist<nearest[label][0]:
                nearest[label]=(dist,(lat0+y,lon0+x/cosine)); feature_meta[label]={'way_id':way.attrib['id'],'tags':tags}
    assert len(nearest)==3, 'Missing proximity feature geometry'
    refs={label:point for label,(_,point) in nearest.items()}
    (RESULTS/'proximity_features.json').write_text(json.dumps(feature_meta,indent=2))
    detail=folium.Map([site.Lat,site.Long],zoom_start=14,tiles='OpenStreetMap')
    distances={}
    for label,(lat,lon) in refs.items():
        p1,p2=math.radians(site.Lat),math.radians(lat); dlat=p2-p1; dlon=math.radians(lon-site.Long)
        km=6371*2*math.asin(math.sqrt(math.sin(dlat/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dlon/2)**2))
        distances[label]=round(km,3)
        folium.PolyLine([[site.Lat,site.Long],[lat,lon]],tooltip=f'{label}: {km:.3f} km',color='blue').add_to(detail)
        folium.Marker([lat,lon],popup=f'{label}: {km:.3f} km straight line').add_to(detail)
    folium.Marker([site.Lat,site.Long],popup='CCAFS SLC-40').add_to(detail); detail.save(ROOT/'site_proximity_map.html')
    print(sites.to_string()); print('Nearest OSM feature straight-line distances (km):',distances)
    return distances


In [2]:
distances=map_analysis()

                    Lat        Long
Launch Site                        
CCAFS LC-40   28.562302  -80.577356
CCAFS SLC-40  28.563197  -80.576820
KSC LC-39A    28.573255  -80.646895
VAFB SLC-4E   34.632834 -120.610745
Nearest OSM feature straight-line distances (km): {'Road': 0.195, 'Coastline': 0.856, 'Railway': 0.0}
